# Terzaghi 2D Multi-Layer

Demo for the 2D Multi layer FEM model to get settlement.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import matplotlib.animation as animation
from IPython.display import HTML

%load_ext autoreload
%autoreload 2
import os
import sys
project_root = os.path.abspath(os.path.join('..'))
sys.path.insert(0, project_root)

from src.geotech_consolidation.models.terzaghi_2d.fem import Get_terzaghi2D_FEA
from src.plotting.terzaghi_2d.plot import Get_Settlement_Animation

H = 5.0
W = H
nx = 25
load = 100.0
base = 2.0
T = 3650 * 24 * 60 * 60
time_steps = 300

# Single-layer demo:
# depths = None
# Cv = 2e-7
# Mv = 5e-4

# Layered demo:
depths = [1.5, 3.0, 5.0]
Cv = [2e-7, 1e-7, 3e-7]
Mv = [5e-4, 8e-4, 4e-4]

time_days = np.linspace(0, T / (60 * 60 * 24), num=time_steps)

## Solve the 2D Model

In [ ]:
settlement_surface, u_hist, unique_X, node_X, node_Y = Get_terzaghi2D_FEA( H=H, W=W, nx=nx, load=load, final_time=T, time_steps=time_steps, Cv=Cv, Mv=Mv, base=base,depths=depths)

print(f"u_hist shape:          {u_hist.shape}          (time_steps x n_nodes)")
print(f"settlement_surface:    {settlement_surface.shape}      (time_steps x nX)")
print(f"Max settlement:        {settlement_surface.max() * 1000:.2f} mm")

fig, anim = Get_Settlement_Animation(settlement_surface, unique_X, time_days, interval = 120)
HTML(anim.to_jshtml())

## Settlement Through Time

Animated settlement profile across the surface width.

In [1]:
fig, ax = plt.subplots(figsize=(7, 4))
line, = ax.plot(unique_X, -settlement_surface[0], label="Surface settlement")
ax.set_xlim(unique_X.min(), unique_X.max())
ax.set_ylim(-settlement_surface.max() * 1.1, -settlement_surface.min() * 1.1)
ax.set_xlabel("x (m)")
ax.set_ylabel("Settlement (m)")
ax.legend()

def update_settlement(frame):
    line.set_ydata(-settlement_surface[frame])
    ax.set_title(f"Settlement at t = {time_days[frame]:.1f} days")
    return (line,)

anim_settlement = animation.FuncAnimation(
    fig=fig,
    func=update_settlement,
    frames=len(time_days),
    interval=40,
    blit=True,
)

plt.close(fig)
HTML(anim_settlement.to_jshtml())